[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anandjha90/Advanced-Python-for-GenAI-Prep-Training/blob/main/notebooks/05_Deep_Learning.ipynb)

# Module 5: Deep Learning (35 min)

Deep Learning (DL) is machine learning with neural networks that have many layers. Classical ML works best on tables with hand-made features. Deep learning learns the features itself from raw data, which is why it dominates images, audio and text, and why every LLM is a deep neural network.

## 5.1 The artificial neuron

A neuron multiplies each input by a weight, adds them up with a bias, and passes the result through an activation function:

**z = w1x1 + w2x2 + ... + wnxn + b = w . x + b, output = activation(z)**

The weights say how important each input is, and the bias shifts the decision boundary. Learning means finding good weights and biases. A single neuron with a sigmoid activation is exactly logistic regression.

![Figure 9: A single neuron predicting the probability that a shopper clicks Buy.](https://raw.githubusercontent.com/anandjha90/Advanced-Python-for-GenAI-Prep-Training/main/images/d09_neuron.png)
*Figure 9: A single neuron predicting the probability that a shopper clicks Buy.*

In [ ]:
import numpy as np

# A single artificial neuron: z = w . x + b, output = activation(z)
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# Real-world example: will this customer click "Buy"?
# features x = [items in cart, minutes on page, has coupon (0/1)]
x = np.array([3, 7.5, 1])
w = np.array([0.4, 0.15, 1.2])     # learned importance of each feature
b = -2.5                            # learned bias (baseline tendency)

z = np.dot(w, x) + b
p = sigmoid(z)
print(f"z = w.x + b = {z:.3f}")
print(f"P(buy) = sigmoid(z) = {p:.3f} -> {'BUY' if p >= 0.5 else 'NO BUY'}")

# The same neuron applied to many customers at once (matrix form)
X = np.array([[3, 7.5, 1],
              [0, 1.0, 0],
              [1, 3.0, 1],
              [5, 12.0, 0]])
probs = sigmoid(X @ w + b)
for row, pr in zip(X, probs):
    print(f"cart={row[0]:.0f}, minutes={row[1]:>4}, coupon={row[2]:.0f} -> P(buy)={pr:.2f}")

## 5.2 Activation functions and softmax

Without a non-linear activation, stacking layers would still produce a straight-line model. Activations let networks learn curves and complex patterns.

| Activation | Output range | Where it is used |
|---|---|---|
| Sigmoid | 0 to 1 | Output layer for yes/no (binary) problems |
| Tanh | -1 to 1 | Hidden layers of older networks, RNNs |
| ReLU = max(0, z) | 0 to infinity | Default for hidden layers: simple and fast to train |
| Leaky ReLU / GELU | Small negative values allowed | GELU is used inside Transformers such as GPT and BERT |
| Softmax | Probabilities that sum to 1 | Output layer for multi-class problems; next-word prediction in LLMs |


> **Real-world example**
>
> When an LLM writes, its last layer produces a score (logit) for every word in its vocabulary, and softmax turns these into probabilities. The **temperature** setting in every LLM API divides the logits before softmax: low temperature makes the top word dominate (focused answers), high temperature flattens the probabilities (creative answers).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

z = np.linspace(-6, 6, 200)
activations = {
    "Sigmoid (0 to 1)": 1 / (1 + np.exp(-z)),
    "Tanh (-1 to 1)": np.tanh(z),
    "ReLU max(0, z)": np.maximum(0, z),
    "Leaky ReLU": np.where(z > 0, z, 0.1 * z),
}
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, (name, values) in zip(axes, activations.items()):
    ax.plot(z, values, linewidth=2)
    ax.axhline(0, color="grey", linewidth=0.5); ax.axvline(0, color="grey", linewidth=0.5)
    ax.set_title(name)
plt.tight_layout()
plt.show()

# Softmax: turns raw scores (logits) into probabilities that sum to 1.
# This is exactly how an LLM picks the next word from its vocabulary.
def softmax(logits, temperature=1.0):
    scaled = np.array(logits) / temperature
    e = np.exp(scaled - scaled.max())          # subtract max for numerical stability
    return e / e.sum()

words = ["delivered", "late", "broken", "excellent"]
logits = [2.0, 1.0, 0.1, 1.5]
for t in [0.5, 1.0, 2.0]:
    probs = softmax(logits, temperature=t)
    print(f"temperature={t}: " + ", ".join(f"{w}={p:.2f}" for w, p in zip(words, probs)))
print("Low temperature = confident/deterministic, high temperature = more random/creative")

## 5.3 Loss functions and gradient descent

- A **loss function** measures how wrong the predictions are: **Mean Squared Error** for regression, **binary cross-entropy** for yes/no, **categorical cross-entropy** for multi-class (and for LLM training).
- **Gradient descent** reduces the loss step by step: compute the gradient (slope) of the loss for each weight and move the weight a little in the opposite direction: **w = w - learning_rate x gradient**.
- The **learning rate** controls the step size. Too small: very slow. Too large: the loss explodes.
- **Epoch** = one pass over the whole training set. **Batch** = a small group of samples used for one update (mini-batch gradient descent).
- **Adam** is the most used optimiser: gradient descent with momentum and an adaptive learning rate per weight.

![Figure 10: Gradient descent walks downhill on the loss surface (left); the learning rate decides whether it is slow, fast or unstable (right).](https://raw.githubusercontent.com/anandjha90/Advanced-Python-for-GenAI-Prep-Training/main/images/d11_gradient.png)
*Figure 10: Gradient descent walks downhill on the loss surface (left); the learning rate decides whether it is slow, fast or unstable (right).*


> **Real-world example**
>
> A marketing team wants to know how revenue responds to ad spend. We learn the line revenue = w x spend + b from scratch with gradient descent and see how the learning rate changes the result. The same loop, at a much larger scale, trains every neural network.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Real-world example: learn  revenue = w * ad_spend + b  with gradient descent
rng = np.random.default_rng(0)
ad_spend = rng.uniform(0, 10, 100)                  # in Rs lakh
revenue = 3.0 * ad_spend + 5 + rng.normal(0, 2, 100)

def train(lr, epochs=1000):
    w, b = 0.0, 0.0
    losses = []
    n = len(ad_spend)
    for _ in range(epochs):
        pred = w * ad_spend + b
        error = pred - revenue
        loss = (error ** 2).mean()                  # Mean Squared Error
        dw = 2 / n * (error * ad_spend).sum()       # dLoss/dw
        db = 2 / n * error.sum()                    # dLoss/db
        w -= lr * dw                                # step downhill
        b -= lr * db
        losses.append(loss)
        if loss > 1e8:                              # stop if training blows up
            break
    return w, b, losses

for lr in [0.001, 0.005, 0.02, 0.04]:
    w, b, losses = train(lr)
    if losses[-1] > 1e8:
        print(f"lr={lr:<6} -> DIVERGED: learning rate too high, the loss exploded")
    else:
        print(f"lr={lr:<6} -> w={w:.2f}, b={b:.2f}, final loss={losses[-1]:.2f}")
    plt.plot(losses, label=f"lr={lr}")

print("True values: w=3.00, b=5.00")
plt.yscale("log"); plt.ylim(1, 1e4); plt.xlabel("Epoch"); plt.ylabel("MSE loss (log scale)")
plt.title("Gradient descent: effect of learning rate"); plt.legend()
plt.show()

## 5.4 Multi-layer networks and backpropagation

A neural network stacks layers: an **input layer** (features), one or more **hidden layers**, and an **output layer**. Training repeats four steps:

1. **Forward pass**: compute predictions layer by layer (Z = XW + b, then activation).
2. **Loss**: compare predictions with the true labels.
3. **Backpropagation**: use the chain rule of calculus to compute how much every weight contributed to the error, moving from the output back to the input.
4. **Update**: change every weight with gradient descent.

![Figure 11: A multi-layer neural network with the forward pass and backpropagation.](https://raw.githubusercontent.com/anandjha90/Advanced-Python-for-GenAI-Prep-Training/main/images/d10_network.png)
*Figure 11: A multi-layer neural network with the forward pass and backpropagation.*


The XOR problem below (output 1 only when the two inputs differ) cannot be solved by a single neuron because no straight line separates the classes. A small hidden layer solves it. Here forward pass and backpropagation are written in pure NumPy so every step is visible.

In [ ]:
import numpy as np

# XOR cannot be solved by one neuron (not linearly separable) but a tiny
# 2-layer network can. This is forward pass + backpropagation in pure NumPy.
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y = np.array([[0], [1], [1], [0]])

rng = np.random.default_rng(42)
W1 = rng.normal(0, 1, (2, 4)); b1 = np.zeros((1, 4))     # input(2) -> hidden(4)
W2 = rng.normal(0, 1, (4, 1)); b2 = np.zeros((1, 1))     # hidden(4) -> output(1)
sigmoid = lambda z: 1 / (1 + np.exp(-z))
lr = 0.5

for epoch in range(10001):
    # Forward pass
    h = np.tanh(X @ W1 + b1)
    out = sigmoid(h @ W2 + b2)
    loss = -np.mean(y * np.log(out + 1e-9) + (1 - y) * np.log(1 - out + 1e-9))

    # Backward pass (chain rule)
    d_out = (out - y) / len(X)                 # gradient of BCE + sigmoid
    dW2 = h.T @ d_out
    db2 = d_out.sum(axis=0, keepdims=True)
    d_h = (d_out @ W2.T) * (1 - h ** 2)        # derivative of tanh
    dW1 = X.T @ d_h
    db1 = d_h.sum(axis=0, keepdims=True)

    # Gradient descent update
    W2 -= lr * dW2; b2 -= lr * db2
    W1 -= lr * dW1; b1 -= lr * db1

    if epoch % 2000 == 0:
        print(f"epoch {epoch:5d}  loss {loss:.4f}")

print("\nPredictions after training:")
for inp, p in zip(X, out.ravel()):
    print(f"  {inp} -> {p:.3f} -> class {int(p > 0.5)}")

## 5.5 Building networks with Keras

Keras (inside TensorFlow) builds the same networks in a few lines:

- `keras.Sequential([...])` stacks layers; `Dense` is a fully connected layer.
- `Dropout(0.2)` randomly switches off 20% of neurons during training to reduce overfitting.
- `compile()` chooses the optimiser, loss and metrics; `fit()` trains; `evaluate()` tests; `predict()` gives probabilities.
- `EarlyStopping` stops training when the validation loss stops improving and restores the best weights.

> **Real-world example**
>
> MNIST contains 70,000 real handwritten digits (28 x 28 pixels). It is the "hello world" of deep learning and the basis of systems that read cheques, forms and postal codes. A simple network reaches about 98% accuracy in under a minute on Colab.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(42)

# Real dataset: 70,000 handwritten digits, 28x28 pixels
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0          # scale pixels to 0-1
print("Train:", x_train.shape, "Test:", x_test.shape)

model = keras.Sequential([
    keras.Input(shape=(28, 28)),
    layers.Flatten(),                         # 28x28 -> 784 vector
    layers.Dense(128, activation="relu"),     # hidden layer 1
    layers.Dropout(0.2),                      # regularisation: randomly switch off 20% of neurons
    layers.Dense(64, activation="relu"),      # hidden layer 2
    layers.Dense(10, activation="softmax"),   # 10 classes -> probabilities
])
model.summary()

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

early_stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)
history = model.fit(x_train, y_train, epochs=10, batch_size=128,
                    validation_split=0.1, callbacks=[early_stop], verbose=2)

test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"\nTest accuracy: {test_acc:.4f}")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history.history["loss"], label="train")
ax[0].plot(history.history["val_loss"], label="validation")
ax[0].set_title("Loss"); ax[0].legend()
ax[1].plot(history.history["accuracy"], label="train")
ax[1].plot(history.history["val_accuracy"], label="validation")
ax[1].set_title("Accuracy"); ax[1].legend()
plt.show()

# Predict 5 test images
probs = model.predict(x_test[:5], verbose=0)
fig, axes = plt.subplots(1, 5, figsize=(10, 2.5))
for ax, img, pr, true in zip(axes, x_test[:5], probs, y_test[:5]):
    ax.imshow(img, cmap="gray_r"); ax.axis("off")
    ax.set_title(f"pred {pr.argmax()} ({pr.max():.0%})\ntrue {true}", fontsize=9)
plt.show()

## 5.6 Convolutional Neural Networks (CNNs)

A dense layer treats an image as a flat list of pixels and ignores which pixels are neighbours. A **convolutional layer** slides small filters (for example 3 x 3) across the image to detect local patterns: edges first, then strokes and shapes in deeper layers. **Max pooling** shrinks the image while keeping the strongest signals. CNNs need far fewer parameters and are more accurate on images.

> **Real-world example**
>
> E-commerce companies use CNNs for visual search ("find similar products from a photo"), detecting damaged items in return photos, and checking that product images meet listing guidelines.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(42)
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train = (x_train / 255.0)[..., None]          # add channel axis -> (60000, 28, 28, 1)
x_test = (x_test / 255.0)[..., None]

cnn = keras.Sequential([
    keras.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, kernel_size=3, activation="relu"),   # 32 filters learn edges/strokes
    layers.MaxPooling2D(),                                 # downsample 2x
    layers.Conv2D(64, kernel_size=3, activation="relu"),   # 64 filters learn shapes
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dropout(0.3),
    layers.Dense(10, activation="softmax"),
])
cnn.summary()
cnn.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
cnn.fit(x_train, y_train, epochs=3, batch_size=128, validation_split=0.1, verbose=2)
print(f"\nCNN test accuracy: {cnn.evaluate(x_test, y_test, verbose=0)[1]:.4f}")
# Tip: in Colab use Runtime > Change runtime type > T4 GPU to train CNNs much faster

## 5.7 Embeddings and attention: the bridge to Transformers and LLMs

- An **embedding layer** is a lookup table that maps each token id to a learnable dense vector. During training, words used in similar contexts get similar vectors, which is the meaning-aware version of the TF-IDF vectors from Module 3.
- **Self-attention** lets every token look at every other token and decide which ones matter for its meaning. In "the phone battery died", the word "died" needs "battery" to be understood.
- Each token embedding is projected into a **Query** (what am I looking for?), a **Key** (what do I contain?) and a **Value** (what information do I pass on?).
- **Attention(Q, K, V) = softmax(Q K^T / sqrt(d_k)) V**: dot products of queries and keys give relevance scores, softmax turns them into weights, and the weights mix the values.
- A **Transformer** stacks many attention + dense layers. GPT, Claude, Gemini and Llama are large Transformers trained to predict the next token.

![Figure 12: Scaled dot-product attention, the core operation of every Transformer.](https://raw.githubusercontent.com/anandjha90/Advanced-Python-for-GenAI-Prep-Training/main/images/d12_attention.png)
*Figure 12: Scaled dot-product attention, the core operation of every Transformer.*


Everything in this formula was covered today: dot products and matrix multiplication (Module 3), softmax and probability (Modules 2 and 5), and learned weights (Module 5).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras import layers

# 1) Embedding layer: maps each token id to a learnable dense vector
embedding = layers.Embedding(input_dim=1000, output_dim=8)     # vocab 1000, 8-D vectors
token_ids = np.array([[12, 45, 7, 3]])                          # one sentence of 4 tokens
vectors = embedding(token_ids).numpy()
print("Token ids shape:", token_ids.shape, "-> embeddings shape:", vectors.shape)

# 2) Scaled dot-product attention - the core operation inside every Transformer/LLM
#    Attention(Q, K, V) = softmax(Q K^T / sqrt(d_k)) V
def softmax(x):
    e = np.exp(x - x.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

tokens = ["the", "phone", "battery", "died"]
rng = np.random.default_rng(1)
d_model = 6
X = rng.normal(0, 1, (4, d_model))                 # pretend these are the 4 token embeddings
X[2] = X[1] * 0.9 + rng.normal(0, 0.2, d_model)    # make "battery" related to "phone"
X[3] = X[2] * 0.8 + rng.normal(0, 0.3, d_model)    # and "died" related to "battery"

Wq, Wk, Wv = (rng.normal(0, 0.5, (d_model, d_model)) for _ in range(3))
Q, K, V = X @ Wq, X @ Wk, X @ Wv                   # queries, keys, values

scores = Q @ K.T / np.sqrt(d_model)                # how much each token "looks at" others
weights = softmax(scores)                          # each row sums to 1
output = weights @ V                               # context-aware token representations

print("\nAttention weights (rows = query token):")
print("         " + "".join(f"{t:>9}" for t in tokens))
for t, row in zip(tokens, weights):
    print(f"{t:>8} " + "".join(f"{w:9.2f}" for w in row))
print("Output shape:", output.shape, "(same as input: 4 tokens x 6 dims)")

plt.imshow(weights, cmap="Blues")
plt.xticks(range(4), tokens); plt.yticks(range(4), tokens)
plt.colorbar(); plt.title("Self-attention weights")
plt.show()

## 5.8 Module 5 practical: review sentiment classifier

We train a neural network on 50,000 real IMDB movie reviews to classify sentiment, then score our own sentences. The model learns a 32-dimensional embedding for every word and averages them to represent a review. The same architecture can classify product reviews, support tickets or chat messages.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# PRACTICAL: review sentiment classifier on 50,000 real IMDB movie reviews
tf.random.set_seed(42)
VOCAB, MAXLEN = 10_000, 200

(x_train, y_train), (x_test, y_test) = keras.datasets.imdb.load_data(num_words=VOCAB)
print("Reviews:", len(x_train), "train /", len(x_test), "test")

# Every review is already a list of word ids; pad/cut them to the same length
x_train = keras.utils.pad_sequences(x_train, maxlen=MAXLEN)
x_test = keras.utils.pad_sequences(x_test, maxlen=MAXLEN)

model = keras.Sequential([
    keras.Input(shape=(MAXLEN,)),
    layers.Embedding(VOCAB, 32),                 # each word -> 32-D learned vector
    layers.GlobalAveragePooling1D(),             # average word vectors -> review vector
    layers.Dense(32, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid"),       # P(positive)
])
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.fit(x_train, y_train, epochs=4, batch_size=256, validation_split=0.2, verbose=2)
print(f"\nTest accuracy: {model.evaluate(x_test, y_test, verbose=0)[1]:.4f}")

# Score our own reviews: convert words to the same ids the dataset uses
word_index = keras.datasets.imdb.get_word_index()
def encode(text):
    ids = [1]                                    # 1 = start token
    for word in text.lower().replace(",", " ").replace(".", " ").split():
        idx = word_index.get(word)
        ids.append(idx + 3 if idx is not None and idx + 3 < VOCAB else 2)   # 2 = unknown word
    return keras.utils.pad_sequences([ids], maxlen=MAXLEN)

for review in ["An excellent and wonderful story, brilliant acting",
               "Boring, waste of time. The worst plot and terrible acting",
               "It was okay, some good moments but too long"]:
    p = float(model.predict(encode(review), verbose=0)[0, 0])
    print(f"{p:.2f} {'POSITIVE' if p > 0.5 else 'NEGATIVE'} : {review}")

> **Discussion**
>
> The model reaches about 86% accuracy with a very simple architecture. Fine-tuned Transformers reach 93 to 95% on this task because attention understands word order and negation ("not good"). In GenAI projects, the usual choices are calling an LLM with a prompt, using its embeddings plus a small classifier, or fine-tuning.